# 第6回：モデルは本当に当たっているか

**今日の問い：手元のスコアをどこまで信じてよいか。**

上から順に実行してください。`TRY`は全員、`CHANGE`は値を1つ変える練習、
`CHALLENGE`は余裕がある人向けです。`DEEP DIVE`は経験者や自習向けの発展です。
分からないコードは、セル全体ではなく気になる数行をM365 Copilotへ貼って相談します。


In [ ]:
from pathlib import Path

def find_repo_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("pyproject.tomlがある勉強会フォルダ内で実行してください")

ROOT = find_repo_root()
DATA = ROOT / "data"
print("教材フォルダ:", ROOT)


## この回でできるようになること

- 学習・検証・テストの役割を区別し、前処理を分割の内側へ入れる
- 分割方式ごとのスコアのばらつきを比べ、楽観的な評価を見抜く
- ネストした交差検証とadversarial validationで、楽観の少ない推定と分布ずれを確かめる

### 進み方

`CORE`は同期90分で扱う本線、`DEEP DIVE`は時間があれば扱う深掘り、
`SELF-STUDY`は任意自習です。すべて終わらなくても次回へ進めます。
経験者は`CORE`を早めに終え、`DEEP DIVE`を5人で分担して読むと深まります。

### 先に押さえる言葉

- 汎化：未知データでも性能を保つこと
- リーク：予測時には得られない情報が学習へ混ざること
- グループ分割：関連試料を同じ側へまとめる分割
- ネストCV：探索と評価を二重の交差検証で分ける方法
- adversarial validation：学習とテストを見分けられるか調べる手法

> **実行前の30秒予想**：今日の問いに、今の言葉で仮の答えを書いてから始めます。


In [ ]:
import pandas as pd

df = pd.read_csv(DATA / "compound_experiments.csv")
print(f"{len(df)}行 × {len(df.columns)}列")
df.head()


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

features = ["molecular_weight", "logp", "tpsa", "temperature_c", "reaction_time_h"]
clean = df.dropna(subset=features)
X_train, X_valid, y_train, y_valid = train_test_split(clean[features], clean["active"], test_size=0.25, random_state=42, stratify=clean["active"])


## TRY：木の深さと過学習


In [ ]:
rows = []
for depth in [1, 2, 4, 8, None]:
    model = DecisionTreeClassifier(max_depth=depth, random_state=42).fit(X_train, y_train)
    rows.append({
        "max_depth": str(depth),
        "学習スコア": accuracy_score(y_train, model.predict(X_train)),
        "検証スコア": accuracy_score(y_valid, model.predict(X_valid)),
    })
pd.DataFrame(rows).round(3)


## TRY：リークを入れると不自然に良くなる


In [ ]:
leak_features = [*features, "post_assay_signal"]
leaked = df.dropna(subset=leak_features)
Xl_tr, Xl_va, yl_tr, yl_va = train_test_split(leaked[leak_features], leaked["active"], test_size=0.25, random_state=42, stratify=leaked["active"])
leaked_model = DecisionTreeClassifier(max_depth=3, random_state=42).fit(Xl_tr, yl_tr)
print("リーク列ありの検証スコア:", round(accuracy_score(yl_va, leaked_model.predict(Xl_va)), 3))
print("post_assay_signalは測定後の値。計画時の予測には使えません。")


## CORE深掘り：前処理は分割の内側で行う

全データで標準化してから分割すると、検証情報が学習へ漏れます。Pipelineに入れると各分割の内側で学習されます。


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

filled = clean[features].fillna(clean[features].median())
scaler_all = StandardScaler().fit(filled)               # 誤り：全データで学習
leaked_scores = cross_val_score(LogisticRegression(max_iter=1000), scaler_all.transform(filled), clean["active"], cv=5, scoring="f1")

right_pipe = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=1000))
right_scores = cross_val_score(right_pipe, clean[features], clean["active"], cv=5, scoring="f1")
print("全データ前処理(楽観的) F1平均:", round(leaked_scores.mean(), 3))
print("Pipeline内前処理(正しい) F1平均:", round(right_scores.mean(), 3))


## CHALLENGE：化合物系列を跨がせない分割


In [ ]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, valid_idx = next(splitter.split(clean, groups=clean["scaffold_group"]))
print("学習側の系列:", sorted(clean.iloc[train_idx]["scaffold_group"].unique()))
print("検証側の系列:", sorted(clean.iloc[valid_idx]["scaffold_group"].unique()))


## DEEP DIVE：CV方式の比較・ネストCV・分布ずれ

評価は「将来の使われ方」を模擬します。分割方式で楽観度がどう変わるかを見ます。


In [ ]:
from sklearn.model_selection import KFold, StratifiedKFold, GroupKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier

estimator = make_pipeline(SimpleImputer(strategy="median"), DecisionTreeClassifier(max_depth=4, random_state=42))
X_all, y_all, groups = df[features], df["active"], df["scaffold_group"]
schemes = {
    "KFold": cross_val_score(estimator, X_all, y_all, cv=KFold(5, shuffle=True, random_state=42), scoring="f1"),
    "StratifiedKFold": cross_val_score(estimator, X_all, y_all, cv=StratifiedKFold(5, shuffle=True, random_state=42), scoring="f1"),
    "GroupKFold(系列)": cross_val_score(estimator, X_all, y_all, cv=GroupKFold(5), groups=groups, scoring="f1"),
}
pd.DataFrame({name: {"平均": s.mean(), "標準偏差": s.std(), "最低": s.min()} for name, s in schemes.items()}).T.round(3)


### ネストCV：探索と評価を分ける

同じ分割で設定を選び性能も報告すると過大評価します。内側で探索、外側で評価すると楽観が減ります。


In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier

pipe = make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(random_state=42))
param_dist = {
    "randomforestclassifier__n_estimators": [100, 200, 300],
    "randomforestclassifier__max_depth": [3, 4, 6, None],
    "randomforestclassifier__min_samples_leaf": [1, 2, 4],
}
inner = StratifiedKFold(3, shuffle=True, random_state=1)
outer = StratifiedKFold(5, shuffle=True, random_state=2)
search = RandomizedSearchCV(pipe, param_dist, n_iter=8, cv=inner, scoring="f1", random_state=42)
nested = cross_val_score(search, df[features], df["active"], cv=outer, scoring="f1")
print("ネストCVの外側F1:", nested.round(3))
print("楽観の少ない推定 平均±SD:", round(nested.mean(), 3), "±", round(nested.std(), 3))


### adversarial validation：学習とテストは似ているか

学習かテストかを当てる分類器のAUCが高いほど、分布がずれています。


In [ ]:
train_c = pd.read_csv(DATA / "local_competition" / "train.csv")
test_c = pd.read_csv(DATA / "local_competition" / "test.csv")
adv_features = ["temperature_c", "reaction_time_h", "concentration_m", "molecular_weight", "logp", "tpsa"]
combined = pd.concat([
    train_c[adv_features].assign(is_test=0),
    test_c[adv_features].assign(is_test=1),
], ignore_index=True)
adv_model = make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(n_estimators=200, random_state=42))
auc = cross_val_score(adv_model, combined[adv_features], combined["is_test"], cv=5, scoring="roc_auc")
print("adversarial validation AUC:", round(auc.mean(), 3))
print("0.5付近なら分布は近い。0.8以上なら分布ずれを疑う。")


## よくある誤り

- 前処理を全データで済ませてから分割する
- 同じ系列の類似化合物を両側へ入れる
- 検証データを何度も見て実質的に学習する

## SELF-STUDY（任意・30〜60分）

- KFold・StratifiedKFold・GroupKFoldのF1分布を箱ひげ図で比べる
- adversarial validationのAUCを下げる列を1つ見つけ理由を書く

成果は完成したコードでなくても、予想・変更点・出力・解釈を4行で残せば十分です。

## 振り返りチェック

1. 検証とテストの違いは何か
2. ネストCVが必要になるのはどんなときか
3. adversarial validationのAUCが高いと何を意味するか

答えに詰まった項目が、次に見返す場所です。暗記ではなくNotebookの該当セルを指せればOKです。
